# Analysis notebook

Load saved experiment results and produce figures.  
This is the canonical notebook — keep it clean and reproducible.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 11

from src.maps import FAMILIES, compute_lyapunov_general
from src.model import DiscreteTrajectoryTransformer
from src.evaluation import (
    plot_ce_vs_lyapunov, plot_training_curves, plot_trajectory_comparison,
    classify_regime, REGIME_COLORS,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')

# Paths
CACHE_DIR = '../outputs/cache'
FIG_DIR   = '../outputs/figures'
CKPT_DIR  = '../outputs/checkpoints'
RUN_NAME  = 'base'  # change to match your run

## Load results

In [ ]:
import json

with open(f'{CKPT_DIR}/{RUN_NAME}_history.json') as f:
    history = json.load(f)

lya_data = np.load(f'{CACHE_DIR}/lyapunov_grid.npz')
r_grid   = lya_data['r_grid']
lyapunovs = lya_data['lyapunovs']

ce_data  = np.load(f'{CACHE_DIR}/{RUN_NAME}_ce_per_r.npz')
ce_per_r  = ce_data['ce_per_r']
acc_per_r = ce_data['acc_per_r']

print(f'Best epoch: {history["best_epoch"]}  |  best val loss: {history["best_val_loss"]:.4f}')

## Training curves

In [ ]:
plot_training_curves(history)

## CE vs Lyapunov

In [ ]:
plot_ce_vs_lyapunov(r_grid=r_grid, lyapunovs=lyapunovs, ce_per_r=ce_per_r)

## Cross-family generalization

In [ ]:
family_results = {}
for name in ['quadratic', 'tent', 'sine', 'cubic']:
    path = f'{CACHE_DIR}/{RUN_NAME}_ce_{name}.npz'
    data = np.load(path)
    family_results[name] = {'ce': data['ce'], 'acc': data['acc'],
                             'params': FAMILIES[name]['params']}
    print(f'{name:12s}  mean CE: {data["ce"].mean():.4f}')